In [1]:
import pandas as pd
import numpy as np
import re
import html
import unicodedata
import json

from pathlib import Path
from collections import Counter

In [2]:
PROJECT_ROOT = Path.cwd().parent if (Path.cwd().parent / "dataset").exists() else Path.cwd()

RAW_CANDIDATES = [
    PROJECT_ROOT / "dataset" / "raw" / "New_Attack_Dataset.csv",
    PROJECT_ROOT / "dataset" / "New_Attack_Dataset.csv",
]

RAW_PATH = None
for path in RAW_CANDIDATES:
    if path.exists():
        RAW_PATH = path
        break

if RAW_PATH is None:
    raise FileNotFoundError(
        "Không tìm thấy New_Attack_Dataset.csv. "
        "Hãy đặt file vào dataset/raw/ hoặc dataset/."
    )

PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = PROCESSED_DIR / "attack_dataset_processed.csv"
REPORT_PATH = RESULTS_DIR / "preprocessing_report.json"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_PATH:", RAW_PATH)
print("OUTPUT_PATH:", OUTPUT_PATH)
print("REPORT_PATH:", REPORT_PATH)

PROJECT_ROOT: d:\Truong\FPT\SUMMER2026\AIC211\CTI_ATT&CK\cti_attck
RAW_PATH: d:\Truong\FPT\SUMMER2026\AIC211\CTI_ATT&CK\cti_attck\dataset\raw\New_Attack_Dataset.csv
OUTPUT_PATH: d:\Truong\FPT\SUMMER2026\AIC211\CTI_ATT&CK\cti_attck\dataset\processed\attack_dataset_processed.csv
REPORT_PATH: d:\Truong\FPT\SUMMER2026\AIC211\CTI_ATT&CK\cti_attck\results\preprocessing_report.json


In [3]:
df_raw = pd.read_csv(RAW_PATH)

print("Shape:", df_raw.shape)
display(df_raw.head())
print(df_raw.columns.tolist())

Shape: (22399, 2)


,Cleaned_Text,Labels
0,Authentication Bypass via SQL Injection Mobile...,"T1078,T1190"
1,Union-Based SQL Injection AI Agents & LLM Expl...,T1190
2,Error-Based SQL Injection AI Agents & LLM Expl...,T1190
3,Blind SQL Injection AI Agents & LLM Exploits S...,T1190
4,Second-Order SQL Injection AI Agents & LLM Exp...,T1505


['Cleaned_Text', 'Labels']


In [4]:
required_columns = ["Cleaned_Text", "Labels"]

missing_columns = [col for col in required_columns if col not in df_raw.columns]

if missing_columns:
    raise ValueError(f"Dataset thiếu cột: {missing_columns}")

df = df_raw[required_columns].copy()

print("Dataset hợp lệ.")
print("Shape:", df.shape)
display(df.head())

Dataset hợp lệ.
Shape: (22399, 2)


,Cleaned_Text,Labels
0,Authentication Bypass via SQL Injection Mobile...,"T1078,T1190"
1,Union-Based SQL Injection AI Agents & LLM Expl...,T1190
2,Error-Based SQL Injection AI Agents & LLM Expl...,T1190
3,Blind SQL Injection AI Agents & LLM Exploits S...,T1190
4,Second-Order SQL Injection AI Agents & LLM Exp...,T1505


In [5]:
print("Missing values:")
print(df.isna().sum())

print("\nDuplicated rows:")
print(df.duplicated().sum())

print("\nSample labels:")
display(df["Labels"].head(10))

Missing values:
Cleaned_Text    0
Labels          0
dtype: int64

Duplicated rows:
0

Sample labels:


0    T1078,T1190
1          T1190
2          T1190
3          T1190
4          T1505
5          T1048
6          T1190
7          T1505
8          T1190
9    T1190,T1505
Name: Labels, dtype: object

In [6]:
def remove_procedural_noise(text):
    """
    Xóa nhiễu dạng hướng dẫn từng bước, tiêu đề template, cụm lặp lại.
    Không xóa chữ 'step' khi nó nằm trong câu tự nhiên.
    """
    text = str(text)

    # Tách heading Step/Phase bị dính vào từ trước: "logsStep 3" -> "logs. Step 3"
    text = re.sub(
        r"(?<=[a-z)])(?=(?:Step|Phase)\s+\d+\b)",
        ". ",
        text
    )

    # Xóa numbering bị dính sau dấu chấm: "collector.2. Load" -> "collector. Load"
    text = re.sub(
        r"(?<=[A-Za-z)])\.\s*\d{1,2}\.\s*",
        ". ",
        text
    )

    # Xóa số thứ tự dạng "1. Find", "10. Cleanup"
    # Không ăn nhầm IP vì sau dấu chấm phải có khoảng trắng
    text = re.sub(
        r"(?m)(^|\s)\d{1,2}\.\s+",
        " ",
        text
    )

    # Xóa số thứ tự bị dính sau dấu câu: "database.2. Send" -> "database. Send"
    # Không ăn nhầm IP vì trước số phải là chữ hoặc dấu đóng ngoặc, không phải số
    text = re.sub(
        r"(?<=[A-Za-z\)])\d{1,2}\.\s*",
        " ",
        text
    )

    # Chỉ xóa Step/Phase khi có số: Step 1, STEP 10, Phase 2
    text = re.sub(
        r"\b(step|phase)\s+\d+\s*[:\-–—]?",
        " ",
        text,
        flags=re.IGNORECASE
    )

    # Một số heading template bị dính với động từ kế tiếp: "CleanupRemove"
    text = re.sub(r"Cleanup(?=[A-Z])", " ", text)

    boilerplate_phrases = [
        r"\bAI Agents\s*&\s*LLM Exploits\b",
        r"\bAI/ML Security\b",
        r"\bRed Team\b",
        r"\bBlue Team\b",
        r"\bPCAP Dataset\b",
        r"\bBeginner Friendly\b",
        r"\bOptional\b",
        r"\bCleanup\b",
        r"\bSetup Lab\b",
        r"\bLab Setup\b",
        r"\bTools like\b",
        r"\bYou now have\b",
        r"\bThis simulation demonstrates how\b",
        r"\bThis simulation shows how\b",
        r"\bThis simulation demonstrates\b",
        r"\bThis simulation shows\b",
        r"\bGreat for demos\b",
        r"\bGreat for dataset generation\b",
    ]

    for phrase in boilerplate_phrases:
        text = re.sub(
            phrase + r"\s*[.:]?",
            " ",
            text,
            flags=re.IGNORECASE
        )

    # Chuẩn hóa một số cụm bị tách sai
    text = re.sub(r"\bwi\s*fi\b", "wifi", text, flags=re.IGNORECASE)
    text = re.sub(r"\bci\s*cd\b", "cicd", text, flags=re.IGNORECASE)
    text = re.sub(r"\bpost\s+quantum\b", "postquantum", text, flags=re.IGNORECASE)

    # Cleanup dấu câu rỗng
    text = re.sub(r"\s+\.\s+", ". ", text)
    text = re.sub(r"\s+,", ",", text)
    text = re.sub(r"\s+\.", ".", text)
    text = re.sub(r"\.{2,}", ".", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text

In [7]:
def remove_mitre_id_leakage_from_text(text):
    """
    Trung hòa MITRE ATT&CK Technique IDs trong input text để tránh data leakage.
    Không áp dụng cho cột Labels.
    """
    text = str(text)

    # Trung hòa MITRE technique URLs dạng:
    # https://attack.mitre.org/techniques/T1059/
    # https://attack.mitre.org/techniques/T1059/001/
    text = re.sub(
        r"https?://attack\.mitre\.org/techniques/T\d{4}(?:/\d{3})?/?",
        " MITRE_TECHNIQUE_REF ",
        text,
        flags=re.IGNORECASE
    )

    # Trung hòa mọi Technique ID dạng Txxxx hoặc Txxxx.xxx
    text = re.sub(
        r"\bT\d{4}(?:\.\d{3})?\b",
        " MITRE_TECHNIQUE_ID ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(r"\s+", " ", text).strip()

    return text

In [8]:
def add_sqli_signal_tokens(text):
    """
    Thêm các token tín hiệu SQL Injection trước khi làm sạch ký tự đặc biệt.
    Không xóa payload gốc ngay, chỉ append token vào cuối text.
    """
    tokens = []
    text_str = str(text)
    lower_text = text_str.lower()

    has_sqli_context = bool(
        re.search(
            r"(?:\b(?:sql injection|sqli|blind sql|time[- ]based sql|sqlmap|sqlninja|"
            r"union\s+(?:all\s+)?select|information_schema|waitfor\s+delay|pg_sleep)\b|"
            r"@@version|\bselect\b.{0,120}\bfrom\b)",
            lower_text,
            flags=re.IGNORECASE
        )
    )

    # Boolean-based SQLi
    if re.search(r"\b(or|and)\s+['\"]?1['\"]?\s*=\s*['\"]?1['\"]?(?!\w)", text_str, flags=re.IGNORECASE):
        tokens.append("SQLI_BOOLEAN_TRUE")

    if re.search(r"\b(or|and)\s+['\"]?1['\"]?\s*=\s*['\"]?2['\"]?(?!\w)", text_str, flags=re.IGNORECASE):
        tokens.append("SQLI_BOOLEAN_FALSE")

    # SQL comment markers: --, /*, */
    if has_sqli_context and re.search(r"(--|/\*|\*/)", text_str):
        tokens.append("SQLI_COMMENT")

    # Time-based SQLi
    if has_sqli_context and re.search(
        r"\b(sleep|benchmark|pg_sleep|waitfor\s+delay)\s*\(",
        text_str,
        flags=re.IGNORECASE
    ):
        tokens.append("SQLI_TIME_DELAY")

    # UNION-based SQLi
    if re.search(r"\bunion\s+(all\s+)?select\b", text_str, flags=re.IGNORECASE):
        tokens.append("SQLI_UNION_SELECT")

    # SQL enumeration functions
    if has_sqli_context and re.search(
        r"\b(substring|substr|ascii|length|database|version|user|schema_name)\s*\(",
        text_str,
        flags=re.IGNORECASE
    ):
        tokens.append("SQLI_ENUMERATION_FUNC")

    # Database version variable
    if re.search(r"@@version", text_str, flags=re.IGNORECASE):
        tokens.append("SQLI_DB_VERSION")

    # Generic SQLi context
    if has_sqli_context:
        tokens.append("SQLI_CONTEXT")

    if tokens:
        text_str = text_str + " " + " ".join(sorted(set(tokens)))

    return text_str

In [9]:
def normalize_sqli_payloads(text):
    text = str(text)
    has_sqli_context = "SQLI_CONTEXT" in text

    # Conditional time-based SQLi: IF(1=1, SLEEP(5), 0)
    text = re.sub(
        r"\bif\s*\([^)]*(sleep|benchmark|pg_sleep)\s*\([^)]*\)[^)]*\)",
        " SQLI_CONDITIONAL_TIME_DELAY " if has_sqli_context else r"\g<0>",
        text,
        flags=re.IGNORECASE
    )

    # Boolean-based SQLi
    text = re.sub(
        r"\b(or|and)\s+['\"]?1['\"]?\s*=\s*['\"]?1['\"]?(?!\w)",
        " SQLI_BOOLEAN_TRUE ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\b(or|and)\s+['\"]?1['\"]?\s*=\s*['\"]?2['\"]?(?!\w)",
        " SQLI_BOOLEAN_FALSE ",
        text,
        flags=re.IGNORECASE
    )

    # Time-based SQLi
    text = re.sub(
        r"\b(sleep|benchmark|pg_sleep)\s*\([^)]*\)",
        " SQLI_TIME_DELAY " if has_sqli_context else r"\g<0>",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\bwaitfor\s+delay\b",
        " SQLI_TIME_DELAY " if has_sqli_context else r"\g<0>",
        text,
        flags=re.IGNORECASE
    )

    # UNION SELECT
    text = re.sub(
        r"\bunion\s+(all\s+)?select\b",
        " SQLI_UNION_SELECT ",
        text,
        flags=re.IGNORECASE
    )

    # Enumeration functions
    text = re.sub(
        r"\b(substring|substr|ascii|length|database|version|user|schema_name)\s*\([^)]*\)",
        " SQLI_ENUMERATION_FUNC " if has_sqli_context else r"\g<0>",
        text,
        flags=re.IGNORECASE
    )

    # Database version variable
    text = re.sub(
        r"@@version",
        " SQLI_DB_VERSION ",
        text,
        flags=re.IGNORECASE
    )

    # SQL comment markers
    text = re.sub(
        r"(--|/\*|\*/)",
        " SQLI_COMMENT " if has_sqli_context else r"\g<0>",
        text
    )

    # Mỗi signal chỉ cần xuất hiện một lần; tránh token vừa replace vừa append bị lặp.
    seen_signals = set()

    def keep_first_signal(match):
        token = match.group(0)
        if token in seen_signals:
            return " "
        seen_signals.add(token)
        return token

    text = re.sub(r"\bSQLI_[A-Z_]+\b", keep_first_signal, text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [10]:
def normalize_cve(text):
    """
    CVE-2021-44228 -> CVE_TOKEN CVE_YEAR_2021
    Giữ tín hiệu có CVE và giữ năm, không để model học vẹt mã CVE cụ thể.
    """
    def repl(match):
        cve = match.group(0).upper()
        year = cve.split("-")[1]
        return f" CVE_TOKEN CVE_YEAR_{year} "

    return re.sub(
        r"\bCVE-\d{4}-\d{4,7}\b",
        repl,
        str(text),
        flags=re.IGNORECASE
    )

In [11]:
def normalize_windows_path(text):
    """
    Chuẩn hóa Windows path nhưng giữ tín hiệu quan trọng.
    Bản này xử lý được path có dấu cách như Start Menu, Program Files.
    """
    text = str(text)

    # Bảo vệ các folder Windows có dấu cách để regex không bị cắt giữa chừng
    protected_phrases = {
        "Start Menu": "Start_Menu",
        "Program Files (x86)": "Program_Files_x86",
        "Program Files": "Program_Files",
        "Common Files": "Common_Files",
    }

    for original, protected in protected_phrases.items():
        text = re.sub(
            re.escape(original),
            protected,
            text,
            flags=re.IGNORECASE
        )

    def repl(match):
        path = match.group(0)
        path_lower = path.lower()

        tokens = ["WINDOWS_PATH"]

        if "system32" in path_lower:
            tokens.append("system32")

        if "syswow64" in path_lower:
            tokens.append("syswow64")

        if "startup" in path_lower:
            tokens.append("startup_folder")

        if "appdata" in path_lower:
            tokens.append("appdata")

        if "temp" in path_lower:
            tokens.append("temp_dir")

        if "programdata" in path_lower:
            tokens.append("programdata")

        if "program_files" in path_lower:
            tokens.append("program_files")

        # Lấy basename cuối path
        basename = re.split(r"[\\/]", path)[-1]
        basename = basename.strip().lower()

        if basename and "." in basename:
            tokens.append(basename)

        return " " + " ".join(dict.fromkeys(tokens)) + " "

    # Chuẩn hóa UNC path (\\server\share\file) nhưng giữ basename có extension
    def repl_unc(match):
        path = match.group(0)
        basename = re.split(r"[\\/]", path)[-1].strip().lower()
        tokens = ["UNC_PATH", "network_share"]
        if basename and "." in basename:
            tokens.append(basename)
        return " " + " ".join(dict.fromkeys(tokens)) + " "

    text = re.sub(
        r"(?<!\\)\\\\[A-Za-z0-9_.-]+\\[A-Za-z0-9$_.-]+(?:\\[^\s\"']*)?",
        repl_unc,
        text
    )

    # Chuẩn hóa drive root đứng riêng như C:\ hoặc Z:\
    text = re.sub(
        r"(?<![A-Za-z0-9])[A-Za-z]:\\(?=\s|$|[;,)])",
        " WINDOWS_PATH drive_root ",
        text
    )

    # Bắt Windows path sau khi đã bảo vệ dấu cách
    text = re.sub(
        r"[A-Za-z]:\\[^\s\"']+",
        repl,
        text
    )

    return text

In [12]:
def normalize_unix_path(text):
    """
    Chuẩn hóa Unix/Linux path thật, tránh ăn nhầm Windows command options như:
    /c, /all, /node, /quiet, /S, /Q.
    """
    text = str(text)

    important_paths = {
        "/root/.ssh/authorized_keys": "UNIX_PATH ssh_authorized_keys",
        "/etc/passwd": "UNIX_PATH etc_passwd",
        "/etc/shadow": "UNIX_PATH etc_shadow",
        "/etc/cron": "UNIX_PATH cron_path",
        "/var/log": "UNIX_PATH log_dir",
        "/tmp": "UNIX_PATH tmp_dir",
        "/home": "UNIX_PATH home_dir",
    }

    for path, token in sorted(important_paths.items(), key=lambda x: len(x[0]), reverse=True):
        text = text.replace(path, f" {token} ")

    unix_root_dirs = r"(etc|var|tmp|root|home|usr|bin|sbin|opt|dev|proc|sys|lib|lib64|mnt|media|srv)"

    text = re.sub(
        rf"(?<!\w)/{unix_root_dirs}(?:/[A-Za-z0-9._-]+)*",
        " UNIX_PATH ",
        text
    )

    return text


In [13]:
def normalize_registry_key(text):
    """
    Chuẩn hóa Registry nhưng giữ tín hiệu Run key, Services key nếu có.
    """
    def repl(match):
        key = match.group(0)
        key_lower = key.lower()

        tokens = ["REGISTRY_KEY"]

        if "\\run" in key_lower or "\\runonce" in key_lower:
            tokens.append("run_key")

        if "currentversion" in key_lower:
            tokens.append("currentversion")

        if "services" in key_lower:
            tokens.append("services_key")

        return " " + " ".join(dict.fromkeys(tokens)) + " "

    return re.sub(
        r"\b(HKLM|HKCU|HKEY_LOCAL_MACHINE|HKEY_CURRENT_USER)\\[^\s\"']+",
        repl,
        str(text),
        flags=re.IGNORECASE
    )

In [14]:
def normalize_cti_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Decode HTML entities nếu có
    text = html.unescape(text)

    # Chuẩn hóa unicode
    text = unicodedata.normalize("NFKC", text)

    #tránh bị leak nhãn trong cột text
    text = remove_mitre_id_leakage_from_text(text)

    # Xóa nhiễu procedural/template trước
    text = remove_procedural_noise(text)

    # Thêm token tín hiệu SQLi
    text = add_sqli_signal_tokens(text)

    # Chuẩn hóa payload SQLi thành token
    text = normalize_sqli_payloads(text)


    # Chuẩn hóa CVE nhưng giữ năm
    text = normalize_cve(text)

    # Chuẩn hóa URL
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " URL_TOKEN ",
        text,
        flags=re.IGNORECASE
    )

    # Chuẩn hóa email
    text = re.sub(
        r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
        " EMAIL_TOKEN ",
        text
    )

    # Chuẩn hóa IPv4
    text = re.sub(
        r"\b(?:\d{1,3}\.){3}\d{1,3}\b",
        " IPV4_TOKEN ",
        text
    )

    # Chuẩn hóa hash MD5/SHA1/SHA256
    text = re.sub(
        r"\b[a-fA-F0-9]{32}\b|\b[a-fA-F0-9]{40}\b|\b[a-fA-F0-9]{64}\b",
        " HASH_TOKEN ",
        text
    )

    # Registry trước Windows path để tránh regex path ăn mất
    text = normalize_registry_key(text)

    # Chuẩn hóa path
    text = normalize_windows_path(text)
    text = normalize_unix_path(text)

    # Xóa emoji / ký hiệu trang trí phổ biến
    text = re.sub(r"[✅🟢🔺⚠️👉➡️🧠🛑📌•]+", " ", text)

    # Xóa markdown heading / separator
    text = re.sub(r"#{1,6}", " ", text)
    text = re.sub(r"-{3,}", " ", text)
    text = re.sub(r"_{3,}", " ", text)

    # Xóa nhãn bước còn sót lại
    text = re.sub(
        r"\b(step|phase)\s*\d+\b",
        " ",
        text,
        flags=re.IGNORECASE
    )

    # Xóa quote/backtick/smart quotes sau khi đã giữ SQLi signals
    text = text.replace("`", " ")
    text = text.replace('"', " ")
    text = text.replace("'", " ")
    text = text.replace("“", " ")
    text = text.replace("”", " ")
    text = text.replace("‘", " ")
    text = text.replace("’", " ")

    # Chạy lại procedural cleanup vì việc bỏ quote/thay token có thể làm lộ numbering bị dính.
    text = remove_procedural_noise(text)

    # Cleanup dấu câu rỗng do xóa template để lại
    text = re.sub(r"\s+\.\s+", ". ", text)
    text = re.sub(r"\s+,", ",", text)
    text = re.sub(r"\s+\.", ".", text)
    text = re.sub(r"\.{2,}", ".", text)

    # Xóa khoảng trắng dư
    text = re.sub(r"\s+", " ", text).strip()

    # Xóa dấu câu ở đầu/cuối nếu bị dư
    text = text.strip(" .,-;:")

    return text

In [15]:
def normalize_labels(label_str):
    if pd.isna(label_str):
        return ""

    label_str = str(label_str)

    # Tách bằng dấu phẩy hoặc chấm phẩy
    parts = re.split(r"[,;]", label_str)

    cleaned = []

    for label in parts:
        label = label.strip().upper()

        # Nhận Txxxx hoặc Txxxx.xxx, sau đó gom về parent Txxxx
        match = re.match(r"^(T\d{4})(?:\.\d{3})?$", label)

        if match:
            parent = match.group(1)
            cleaned.append(parent)

    # Xóa trùng nhưng giữ thứ tự xuất hiện
    cleaned = list(dict.fromkeys(cleaned))

    return ",".join(cleaned)


# Cấu hình ánh xạ giảm nhãn cho các mẫu có số lượng nhãn [4, 5, 6, 8, 15] từ notebook 01
RARE_LABEL_MAPPING = {
    "T1021,T1046,T1078,T1210": "T1021,T1046,T1078",
    "T1021,T1040,T1210,T1563": "T1021,T1040,T1563",
    "T1043,T1059,T1071,T1571": "T1059,T1071,T1571",
    "T1048,T1059,T1071,T1090": "T1059,T1071,T1090",
    "T1020,T1048,T1095,T1572": "T1048,T1095,T1572",
    "T1053,T1055,T1547,T1564": "T1053,T1547,T1564",
    "T1021,T1053,T1059,T1105": "T1021,T1059,T1105",
    "T1021,T1059,T1075,T1105": "T1021,T1059,T1105",
    "T1018,T1033,T1057,T1082,T1087": "T1018,T1082,T1087",
    "T1037,T1053,T1204,T1546": "T1037,T1053,T1546",
    "T1202,T1218,T1546,T1556": "T1202,T1546,T1556",
    "T1056,T1071,T1219,T1546": "T1056,T1071,T1219",
    "T1056,T1204,T1546,T1556": "T1056,T1546,T1556",
    "T1078,T1086,T1552,T1557": "T1078,T1086,T1552",
    "T1021,T1071,T1552,T1566": "T1021,T1071,T1552",
    "T1059,T1190,T1203,T1552": "T1059,T1190,T1552",
    "T1078,T1210,T1486,T1530": "T1078,T1486,T1530",
    "T1033,T1041,T1053,T1082": "T1033,T1041,T1082",
    "T1016,T1078,T1210,T1570": "T1078,T1210,T1570",
    "T1059,T1218,T1543,T1569,T1570": "T1543,T1569,T1570",
    "T1041,T1056,T1105,T1113": "T1041,T1056,T1113",
    "T1056,T1078,T1082,T1566": "T1056,T1078,T1082",
    "T1027,T1059,T1078,T1106,T1140,T1548": "T1078,T1140,T1548",
    "T1005,T1057,T1083,T1518": "T1057,T1083,T1518",
    "T1016,T1033,T1057,T1071,T1082,T1573": "T1016,T1033,T1082",
    "T1027,T1070,T1112,T1140": "T1070,T1112,T1140",
    "T1059,T1106,T1204,T1547": "T1059,T1106,T1204",
    "T1053,T1140,T1543,T1574": "T1053,T1543,T1574",
    "T1016,T1027,T1033,T1055,T1071,T1082": "T1027,T1055,T1071",
    "T1003,T1027,T1041,T1047,T1053,T1078,T1112,T1543": "T1003,T1041,T1543",
    "T1041,T1059,T1071,T1105,T1113,T1547": "T1041,T1059,T1113",
    "T1027,T1105,T1204,T1218": "T1105,T1204,T1218",
    "T1016,T1057,T1082,T1083": "T1016,T1057,T1082",
    "T1041,T1057,T1083,T1105,T1113": "T1041,T1057,T1113",
    "T1059,T1071,T1105,T1140": "T1059,T1105,T1140",
    "T1027,T1047,T1055,T1059,T1105,T1218": "T1047,T1055,T1059",
    "T1016,T1033,T1047,T1057,T1082": "T1016,T1057,T1082",
    "T1021,T1055,T1078,T1570": "T1021,T1078,T1570",
    "T1055,T1057,T1059,T1070,T1083,T1105": "T1055,T1059,T1105",
    "T1016,T1059,T1082,T1547": "T1016,T1082,T1547",
    "T1059,T1105,T1106,T1140": "T1059,T1105,T1140",
    "T1016,T1033,T1082,T1518,T1547": "T1016,T1082,T1547",
    "T1057,T1059,T1070,T1083,T1105,T1113": "T1057,T1083,T1113",
    "T1016,T1033,T1057,T1082,T1573": "T1016,T1033,T1082",
    "T1012,T1016,T1057,T1082,T1113": "T1012,T1057,T1113",
    "T1078,T1543,T1569,T1570": "T1078,T1543,T1570",
    "T1003,T1005,T1016,T1047,T1053,T1059,T1068,T1071,T1078,T1082,T1140,T1210,T1547,T1570,T1573": "T1053,T1059,T1078",
    "T1047,T1055,T1082,T1218": "T1047,T1055,T1218",
    "T1027,T1053,T1055,T1059,T1112": "T1027,T1053,T1055",
    "T1056,T1059,T1112,T1113": "T1056,T1059,T1113",
    "T1012,T1055,T1056,T1071,T1090,T1112,T1113,T1548": "T1055,T1071,T1090",
    "T1005,T1012,T1057,T1070,T1112": "T1005,T1057,T1112",
    "T1005,T1041,T1056,T1071": "T1041,T1056,T1071",
    "T1057,T1068,T1190,T1518,T1566": "T1057,T1190,T1566",
}


def update_rare_labels(df, mapping=RARE_LABEL_MAPPING):
    """
    Hàm tự động cập nhật nhãn cho dataframe dựa trên từ điển ánh xạ (giảm nhãn từ notebook 01).
    """
    for old_label, new_label in mapping.items():
        df.loc[df["Labels"] == old_label, "Labels"] = new_label
    return df

In [16]:
sample_sqli = """
Blind SQL Injection. The attacker tries a basic test like ' OR 1=1-- 
and then ' AND 1=2--. They use SUBSTRING(@@version,1,1)='5'-- 
and IF(1=1, SLEEP(5), 0)--. Tools include SQLMap and Burp Suite.
"""

processed_sample = normalize_cti_text(sample_sqli)

print("Before:")
print(sample_sqli)

print("\nAfter:")
print(processed_sample)

Before:

Blind SQL Injection. The attacker tries a basic test like ' OR 1=1-- 
and then ' AND 1=2--. They use SUBSTRING(@@version,1,1)='5'-- 
and IF(1=1, SLEEP(5), 0)--. Tools include SQLMap and Burp Suite.


After:
Blind SQL Injection. The attacker tries a basic test like SQLI_BOOLEAN_TRUE SQLI_COMMENT and then SQLI_BOOLEAN_FALSE. They use SQLI_ENUMERATION_FUNC = 5 and SQLI_CONDITIONAL_TIME_DELAY. Tools include SQLMap and Burp Suite. SQLI_CONTEXT SQLI_DB_VERSION SQLI_TIME_DELAY


In [17]:
test_samples = {
    "SQLi_boolean_time_based": """
    Blind SQL Injection. The attacker tests ' OR 1=1-- and ' AND 1=2--.
    Then uses SUBSTRING(@@version,1,1)='5'-- and IF(1=1, SLEEP(5), 0)--.
    Tools include SQLMap and Burp Suite.
    """,

    "CVE_public_facing_app": """
    The adversary exploited CVE-2021-44228 on a public-facing web application
    to achieve remote code execution. After exploitation, the attacker deployed
    a web shell and executed commands on the server.
    """,

    "Windows_path_persistence": r"""
    The malware copied itself to C:\Users\Admin\AppData\Roaming\Microsoft\Windows\Start Menu\Programs\Startup\update.exe
    and configured persistence. It also launched C:\Windows\System32\cmd.exe to execute commands.
    """,

    "Linux_ssh_authorized_keys": """
    The attacker wrote a public key into /root/.ssh/authorized_keys to maintain access.
    The malware also read /etc/passwd and /etc/shadow before exfiltration.
    """,

    "Registry_run_key": r"""
    The malware created a registry value under HKCU\Software\Microsoft\Windows\CurrentVersion\Run
    to execute payload.exe after user login. This allowed persistence across reboots.
    """,

    "PowerShell_download_payload": """
    The attacker used PowerShell to download a second-stage payload from http://malicious.example/payload.exe.
    The script contacted 192.168.1.50 and executed the downloaded file in memory.
    """,

    "Hash_email_url_ioc": """
    The phishing email from attacker@example.com contained a link to https://evil.example/login.
    The downloaded file had SHA256 hash e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855.
    """,

    "WinRM_lateral_movement": """
    Using valid domain admin credentials, the attacker connected to internal hosts through WinRM.
    Remote PowerShell commands were executed to enumerate users, upload tools, and move laterally.
    """,

    "Ransomware_behavior": """
    The ransomware encrypted files on local drives and network shares.
    It deleted backups, stopped security services, and left a ransom note on the desktop.
    """,

    "Nmap_discovery": """
    The adversary performed network discovery using nmap to scan ports 22, 445, 3389, and 5985.
    Open SMB, SSH, RDP, and WinRM services were identified for later movement.
    """,

    "Credential_dumping": """
    The attacker used Mimikatz to dump credentials from LSASS memory.
    Stolen passwords and hashes were used to access additional systems.
    """,

    "Template_noise": """
    AI Agents & LLM Exploits. Step 1: Setup Lab. Step 2: Attacker scans the victim.
    Step 10: Cleanup. This simulation demonstrates how attackers abuse remote services.
    Tools like nmap, hydra, and Wireshark are used.
    """,

    "CLI_double_dash_not_SQLi": r"""
    Run kape.exe --tsource C:\ --target YaraScan and save to \\server\share\output.json.
    """,

    "Non_SQL_user_parenthesis": """
    The IAM user (roles/editor) can access cloud storage.
    """,

    "Glued_step_numbering": """
    Collect evidence.2. Parse logsStep 3: Review results.
    """
}

for name, sample in test_samples.items():
    print("=" * 120)
    print("SAMPLE:", name)
    print("\nBEFORE:")
    print(sample.strip())

    print("\nAFTER:")
    print(normalize_cti_text(sample))
    print()

SAMPLE: SQLi_boolean_time_based

BEFORE:
Blind SQL Injection. The attacker tests ' OR 1=1-- and ' AND 1=2--.
    Then uses SUBSTRING(@@version,1,1)='5'-- and IF(1=1, SLEEP(5), 0)--.
    Tools include SQLMap and Burp Suite.

AFTER:
Blind SQL Injection. The attacker tests SQLI_BOOLEAN_TRUE SQLI_COMMENT and SQLI_BOOLEAN_FALSE. Then uses SQLI_ENUMERATION_FUNC = 5 and SQLI_CONDITIONAL_TIME_DELAY. Tools include SQLMap and Burp Suite. SQLI_CONTEXT SQLI_DB_VERSION SQLI_TIME_DELAY

SAMPLE: CVE_public_facing_app

BEFORE:
The adversary exploited CVE-2021-44228 on a public-facing web application
    to achieve remote code execution. After exploitation, the attacker deployed
    a web shell and executed commands on the server.

AFTER:
The adversary exploited CVE_TOKEN CVE_YEAR_2021 on a public-facing web application to achieve remote code execution. After exploitation, the attacker deployed a web shell and executed commands on the server

SAMPLE: Windows_path_persistence

BEFORE:
The malware copied

In [18]:
test_fix_samples = {
    "IP_should_not_break": """
    The script contacted 192.168.1.50 and executed the downloaded file in memory.
    """,

    "Windows_path_with_space": r"""
    The malware copied itself to C:\Users\Admin\AppData\Roaming\Microsoft\Windows\Start Menu\Programs\Startup\update.exe
    and launched C:\Windows\System32\cmd.exe.
    """,

    "Template_noise": """
    AI Agents & LLM Exploits. Step 1: Setup Lab. Step 2: Attacker scans the victim.
    Step 10: Cleanup. This simulation demonstrates how attackers abuse remote services.
    Tools like nmap, hydra, and Wireshark are used.
    """
}

for name, sample in test_fix_samples.items():
    print("=" * 100)
    print(name)
    print(normalize_cti_text(sample))

# Regression checks cho các lỗi phát hiện trực tiếp trong attack_dataset_processed.csv
cli_result = normalize_cti_text(test_samples["CLI_double_dash_not_SQLi"])
iam_result = normalize_cti_text(test_samples["Non_SQL_user_parenthesis"])
step_result = normalize_cti_text(test_samples["Glued_step_numbering"])

assert "SQLI_COMMENT" not in cli_result
assert "SQLI_ENUMERATION_FUNC" not in iam_result
assert "UNC_PATH network_share" in cli_result
assert ".2." not in step_result and "Step 3" not in step_result
print("All preprocessing regression checks passed.")

IP_should_not_break
The script contacted IPV4_TOKEN and executed the downloaded file in memory
Windows_path_with_space
The malware copied itself to WINDOWS_PATH startup_folder appdata update.exe and launched WINDOWS_PATH system32 cmd.exe
Template_noise
Attacker scans the victim. attackers abuse remote services. nmap, hydra, and Wireshark are used
All preprocessing regression checks passed.


In [19]:
df_processed = df.copy()

df_processed["Cleaned_Text"] = df_processed["Cleaned_Text"].apply(normalize_cti_text)
df_processed["Labels"] = df_processed["Labels"].apply(normalize_labels)

# Tích hợp giảm nhãn cho các mẫu có số lượng nhãn [4, 5, 6, 8, 15] từ notebook 01
df_processed = update_rare_labels(df_processed, RARE_LABEL_MAPPING)

display(df_processed.head())
print("Shape after preprocessing:", df_processed.shape)

,Cleaned_Text,Labels
0,Authentication Bypass via SQL Injection Mobile...,"T1078,T1190"
1,Union-Based SQL Injection SQL Injection This a...,T1190
2,Error-Based SQL Injection SQL Injection This a...,T1190
3,Blind SQL Injection SQL Injection In Blind SQL...,T1190
4,Second-Order SQL Injection SQL Injection In a ...,T1505


Shape after preprocessing: (22399, 2)


In [20]:
before_rows = len(df_processed)

df_processed = df_processed.drop_duplicates(subset=["Cleaned_Text", "Labels"]).reset_index(drop=True)

after_rows = len(df_processed)

print("Removed duplicated rows:", before_rows - after_rows)
print("Remaining rows:", after_rows)

Removed duplicated rows: 107
Remaining rows: 22292


In [21]:
df_processed["label_count"] = df_processed["Labels"].apply(
    lambda x: len(x.split(",")) if isinstance(x, str) and x.strip() else 0
)

label_count_distribution = df_processed["label_count"].value_counts().sort_index()

print(label_count_distribution)
print("Max labels:", df_processed["label_count"].max())
print("Average labels:", df_processed["label_count"].mean())

label_count
1    20332
2     1647
3      313
Name: count, dtype: int64
Max labels: 3
Average labels: 1.101964830432442


In [22]:
label_counter = Counter()

for label_string in df_processed["Labels"]:
    labels = [label.strip() for label in label_string.split(",") if label.strip()]
    label_counter.update(labels)

label_freq = pd.DataFrame(
    label_counter.items(),
    columns=["Label", "Count"]
).sort_values("Count", ascending=False).reset_index(drop=True)

print("Total unique labels:", len(label_freq))

display(label_freq.head(20))
display(label_freq.tail(20))

Total unique labels: 378


,Label,Count
0,T1027,1447
1,T1059,1382
2,T1140,900
3,T1557,770
4,T1203,755
5,T1055,682
6,T1606,662
7,T1204,578
8,T1078,572
9,T1105,570


,Label,Count
358,T0817,1
359,T1618,1
360,T0854,1
361,T0808,1
362,T0831,1
363,T0832,1
364,T1408,1
365,T1148,1
366,T1424,1
367,T1066,1


In [23]:
df_processed["word_count"] = df_processed["Cleaned_Text"].apply(lambda x: len(str(x).split()))

print(df_processed["word_count"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

display(df_processed[["Cleaned_Text", "Labels", "word_count"]].head())

count    22292.000000
mean       112.879105
std        105.882831
min          1.000000
50%        106.000000
75%        159.000000
90%        241.900000
95%        295.000000
99%        494.090000
max       1081.000000
Name: word_count, dtype: float64


,Cleaned_Text,Labels,word_count
0,Authentication Bypass via SQL Injection Mobile...,"T1078,T1190",184
1,Union-Based SQL Injection SQL Injection This a...,T1190,227
2,Error-Based SQL Injection SQL Injection This a...,T1190,245
3,Blind SQL Injection SQL Injection In Blind SQL...,T1190,334
4,Second-Order SQL Injection SQL Injection In a ...,T1505,325


In [24]:
sample_indices = [0, 1, 2, 3, 4]

for idx in sample_indices:
    if idx < len(df_raw):
        print("=" * 100)
        print(f"ROW {idx}")
        print("\nRAW TEXT:")
        print(df_raw.loc[idx, "Cleaned_Text"][:1000])

        print("\nPROCESSED TEXT:")
        print(df_processed.loc[idx, "Cleaned_Text"][:1000])

        print("\nLABELS:")
        print(df_raw.loc[idx, "Labels"], "=>", df_processed.loc[idx, "Labels"])

ROW 0

RAW TEXT:
Authentication Bypass via SQL Injection Mobile Security SQL Injection (SQLi) A login form fails to validate or sanitize input, allowing attackers to log in as admin without knowing the password. Browser, Burp Suite, SQLMap 1. Reconnaissance: Find a login form on the website (e.g., username and password fields). 2. Test for Injection: Enter a simple payload like ' OR '1'='1 in the username or password field. 3. Analyze Response: If the login succeeds or error message changes, it may be vulnerable. 4. Craft Exploit: Use payloads like: Username: ' OR '1'='1' -- Password: anything. 5. Bypass Authentication: The SQL query behind the scenes becomes: SELECT FROM users WHERE username='' OR '1'='1' -- ' AND password='anything'. This always returns true (1=1), tricking the system to log in without a password.. 6. Access Granted: Attacker gets access to admin or user accounts. Web Login Portals (e.g., banking, admin dashboards, e-commerce) Unsanitized input fields in SQL queries 

In [25]:
print("Shape:", df_processed.shape)

print("\nMissing values:")
print(df_processed[["Cleaned_Text", "Labels"]].isna().sum())

empty_text = df_processed[df_processed["Cleaned_Text"].astype(str).str.strip() == ""]
empty_labels = df_processed[df_processed["Labels"].astype(str).str.strip() == ""]

print("\nEmpty text rows:", len(empty_text))
print("Empty label rows:", len(empty_labels))

display(empty_text.head(10))
display(empty_labels.head(10))

Shape: (22292, 4)

Missing values:
Cleaned_Text    0
Labels          0
dtype: int64

Empty text rows: 0
Empty label rows: 0


,Cleaned_Text,Labels,label_count,word_count


,Cleaned_Text,Labels,label_count,word_count


In [26]:
exact_duplicates = df_processed[
    df_processed.duplicated(subset=["Cleaned_Text", "Labels"], keep=False)
].sort_values(["Cleaned_Text", "Labels"])

print("Exact duplicate rows:", len(exact_duplicates))
display(exact_duplicates[["Cleaned_Text", "Labels"]].head(30))

Exact duplicate rows: 0


,Cleaned_Text,Labels


In [27]:
text_label_conflicts = (
    df_processed
    .groupby("Cleaned_Text")["Labels"]
    .nunique()
    .reset_index(name="unique_label_count")
)

conflict_texts = text_label_conflicts[
    text_label_conflicts["unique_label_count"] > 1
]

print("Number of texts with conflicting labels:", len(conflict_texts))

if len(conflict_texts) > 0:
    conflict_examples = df_processed[
        df_processed["Cleaned_Text"].isin(conflict_texts["Cleaned_Text"])
    ].sort_values("Cleaned_Text")

    display(conflict_examples[["Cleaned_Text", "Labels"]].head(100))
else:
    print("Không có conflict label. Có thể drop duplicate an toàn.")

Number of texts with conflicting labels: 19


,Cleaned_Text,Labels
20952,"After successfully decrypting the C2 URLs, the...","T1071,T1140"
16923,"After successfully decrypting the C2 URLs, the...",T1071
15926,CreateProcessInternalW creates a new process,"T1027,T1106"
15934,CreateProcessInternalW creates a new process,T1106
19377,"Downloading, decrypting, and executing a backd...","T1105,T1140"
14809,"Downloading, decrypting, and executing a backd...",T1105
13931,Shellcode which is used for code injection,T1055
18663,Shellcode which is used for code injection,"T1055,T1484"
20048,The asynchronous procedure places a hook on th...,"T1055,T1106"
15700,The asynchronous procedure places a hook on th...,T1055


In [28]:
def label_set(label_str):
    return set(
        label.strip()
        for label in str(label_str).split(",")
        if label.strip()
    )

def is_subset_superset_conflict(label_list):
    sets = [label_set(x) for x in label_list]

    for i in range(len(sets)):
        for j in range(len(sets)):
            if not (sets[i].issubset(sets[j]) or sets[j].issubset(sets[i])):
                return False

    return True


conflict_summary = (
    conflict_examples
    .groupby("Cleaned_Text")["Labels"]
    .apply(list)
    .reset_index(name="label_versions")
)

conflict_summary["is_subset_superset"] = conflict_summary["label_versions"].apply(
    is_subset_superset_conflict
)

print(conflict_summary["is_subset_superset"].value_counts())

easy_conflicts = conflict_summary[conflict_summary["is_subset_superset"] == True]
hard_conflicts = conflict_summary[conflict_summary["is_subset_superset"] == False]

print("Easy conflicts:", len(easy_conflicts))
print("Hard conflicts:", len(hard_conflicts))

display(easy_conflicts.head(20))
display(hard_conflicts.head(20))

is_subset_superset
True     12
False     7
Name: count, dtype: int64
Easy conflicts: 12
Hard conflicts: 7


,Cleaned_Text,label_versions,is_subset_superset
0,"After successfully decrypting the C2 URLs, the...","[T1071,T1140, T1071]",True
1,CreateProcessInternalW creates a new process,"[T1027,T1106, T1106]",True
2,"Downloading, decrypting, and executing a backd...","[T1105,T1140, T1105]",True
3,Shellcode which is used for code injection,"[T1055, T1055,T1484]",True
4,The asynchronous procedure places a hook on th...,"[T1055,T1106, T1055]",True
5,The backdoor uses an RC4 key configured by an ...,"[T1140,T1573, T1573]",True
6,The banking trojan can collect the victim s lo...,"[T1056,T1082, T1056,T1078,T1082]",True
7,The names of Windows API functions are stored ...,"[T1027,T1106, T1027]",True
8,"To avoid detection, attackers renamed Windows ...","[T1036,T1059, T1036]",True
9,VirtualProtect changes the permissions of the ...,"[T1027,T1106, T1106]",True


,Cleaned_Text,label_versions,is_subset_superset
12,decrypt these database passwords,"[T1078, T1140]",False
13,downloading and executing two other PowerShell...,"[T1059, T1105]",False
14,passes the obfuscated string to a function tha...,"[T1027, T1140]",False
15,steal credentials by decrypting them from regi...,"[T1005, T1078]",False
16,used HTTPS reverse proxies to redirect C2 traffic,"[T1090, T1071]",False
17,whoami /groups,"[T1033, T1016]",False
18,wmic /node,"[T1047, T1070]",False


In [29]:
def label_set(label_str):
    return set(
        label.strip()
        for label in str(label_str).split(",")
        if label.strip()
    )


def merge_label_strings(label_series):
    merged = []

    for label_str in label_series:
        labels = str(label_str).split(",")

        for label in labels:
            label = label.strip()

            if label and label not in merged:
                merged.append(label)

    return ",".join(merged)


def manual_resolve_hard_conflict(text, merged_labels):
    """
    Resolve các hard conflict đã phát hiện.
    Nếu không khớp rule nào thì giữ merged_labels để không mất nhãn.
    """
    t = str(text).lower()

    if "downloading and executing two other powershell" in t:
        return "T1059,T1105"

    if "passes the obfuscated string to a function" in t:
        return "T1027,T1140"

    if "steal credentials by decrypting them from regi" in t:
        return "T1005"

    if "used https reverse proxies to redirect c2 traffic" in t:
        return "T1071,T1090"

    if "whoami /groups" in t:
        return "T1033"

    if "wmic /node" in t:
        return "T1047"

    return merged_labels

In [30]:
df_merged = (
    df_processed
    .groupby("Cleaned_Text", as_index=False)
    .agg({"Labels": merge_label_strings})
)

df_merged["Labels"] = df_merged.apply(
    lambda row: manual_resolve_hard_conflict(
        row["Cleaned_Text"],
        row["Labels"]
    ),
    axis=1
)

# Đảm bảo đồng bộ giảm nhãn sau khi gộp và resolve conflict
df_merged = update_rare_labels(df_merged, RARE_LABEL_MAPPING)

df_merged["label_count"] = df_merged["Labels"].apply(
    lambda x: len([label for label in str(x).split(",") if label.strip()])
)

print("Before merge:", df_processed.shape)
print("After merge:", df_merged.shape)

print("\nLabel count after merge:")
print(df_merged["label_count"].value_counts().sort_index())

print("\nMax labels:", df_merged["label_count"].max())

display(df_merged.head())

Before merge: (22292, 4)
After merge: (22273, 3)

Label count after merge:
label_count
1    20316
2     1650
3      307
Name: count, dtype: int64

Max labels: 3


,Cleaned_Text,Labels,label_count
0,$BASE64 encoded payload,T1027,1
1,$INSTDIR File $INSTDIR\o15bmldpqdxcin.dll File...,T1027,1
2,$LogFile Timeline for File Activity DFIR File ...,T1070,1
3,$url = URL_TOKEN This line sets the value of t...,T1105,1
4,%%i in ( WINDOWS_PATH programdata list.txt) do...,T1070,1


In [31]:
check_conflicts = (
    df_merged
    .groupby("Cleaned_Text")["Labels"]
    .nunique()
    .reset_index(name="unique_label_count")
)

remaining_conflicts = check_conflicts[
    check_conflicts["unique_label_count"] > 1
]

print("Remaining conflicts:", len(remaining_conflicts))

if len(remaining_conflicts) > 0:
    display(
        df_merged[
            df_merged["Cleaned_Text"].isin(remaining_conflicts["Cleaned_Text"])
        ][["Cleaned_Text", "Labels"]]
    )
else:
    print("Không còn conflict label.")

Remaining conflicts: 0
Không còn conflict label.


In [32]:
over_3 = df_merged[df_merged["label_count"] > 3]

print("Rows with more than 3 labels:", len(over_3))

display(over_3[["Cleaned_Text", "Labels", "label_count"]].head(50))

Rows with more than 3 labels: 0


,Cleaned_Text,Labels,label_count


In [33]:
def is_valid_label_string(label_str):
    labels = str(label_str).split(",")

    for label in labels:
        label = label.strip()

        if not re.match(r"^T\d{4}$", label):
            return False

    return True


df_merged["valid_label_format"] = df_merged["Labels"].apply(is_valid_label_string)

invalid_labels = df_merged[df_merged["valid_label_format"] == False]

print("Invalid label rows:", len(invalid_labels))

display(invalid_labels[["Cleaned_Text", "Labels"]].head(50))

Invalid label rows: 0


,Cleaned_Text,Labels


In [34]:
df_final = df_merged[["Cleaned_Text", "Labels"]].copy()

print("Final shape:", df_final.shape)

print("\nFinal label count:")
print(
    df_final["Labels"]
    .apply(lambda x: len(str(x).split(",")))
    .value_counts()
    .sort_index()
)

display(df_final.head())

Final shape: (22273, 2)

Final label count:
Labels
1    20316
2     1650
3      307
Name: count, dtype: int64


,Cleaned_Text,Labels
0,$BASE64 encoded payload,T1027
1,$INSTDIR File $INSTDIR\o15bmldpqdxcin.dll File...,T1027
2,$LogFile Timeline for File Activity DFIR File ...,T1070
3,$url = URL_TOKEN This line sets the value of t...,T1105
4,%%i in ( WINDOWS_PATH programdata list.txt) do...,T1070


In [35]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if (Path.cwd().parent / "dataset").exists() else Path.cwd()

PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = PROCESSED_DIR / "attack_dataset_processed.csv"

df_final.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print("Saved final processed dataset to:", OUTPUT_PATH)
print("Final shape:", df_final.shape)

Saved final processed dataset to: d:\Truong\FPT\SUMMER2026\AIC211\CTI_ATT&CK\cti_attck\dataset\processed\attack_dataset_processed.csv
Final shape: (22273, 2)


In [36]:
from collections import Counter
import pandas as pd

# Nếu bạn đã có df_final thì dùng df_final
# Nếu chưa, thay df_final bằng df_merged hoặc df_processed
df_check = df_final.copy()

label_counter = Counter()

for label_string in df_check["Labels"]:
    labels = [
        label.strip()
        for label in str(label_string).split(",")
        if label.strip()
    ]
    label_counter.update(labels)

label_freq = pd.DataFrame(
    label_counter.items(),
    columns=["Label", "Sample_Count"]
).sort_values("Sample_Count", ascending=False).reset_index(drop=True)

print("Total unique labels:", len(label_freq))
print("Total label occurrences:", label_freq["Sample_Count"].sum())

display(label_freq)

Total unique labels: 378
Total label occurrences: 24537


,Label,Sample_Count
0,T1027,1445
1,T1059,1381
2,T1140,899
3,T1557,770
4,T1203,755
...,...,...
373,T0854,1
374,T1171,1
375,T1387,1
376,T1650,1


In [37]:
from collections import Counter
import pandas as pd

df_work = df_final.copy()

label_counter = Counter()

for label_string in df_work["Labels"]:
    labels = [
        label.strip()
        for label in str(label_string).split(",")
        if label.strip()
    ]
    label_counter.update(labels)

label_freq = pd.DataFrame(
    label_counter.items(),
    columns=["Label", "Sample_Count"]
).sort_values("Sample_Count", ascending=False).reset_index(drop=True)

# Stage 1 chỉ học các nhãn có đủ dữ liệu để chia train/validation/test ổn định.
THRESHOLD = 30

frequent_labels = set(
    label_freq[label_freq["Sample_Count"] >= THRESHOLD]["Label"]
)

rare_labels = set(
    label_freq[label_freq["Sample_Count"] < THRESHOLD]["Label"]
)

print("Total labels:", len(label_freq))
print("Frequent labels:", len(frequent_labels))
print("Rare labels:", len(rare_labels))

display(label_freq.head(20))
display(label_freq[label_freq["Sample_Count"] < THRESHOLD])

Total labels: 378
Frequent labels: 108
Rare labels: 270


,Label,Sample_Count
0,T1027,1445
1,T1059,1381
2,T1140,899
3,T1557,770
4,T1203,755
5,T1055,678
6,T1606,662
7,T1204,578
8,T1078,570
9,T1105,569


,Label,Sample_Count
108,T1621,29
109,T1590,28
110,T1134,28
111,T1098,27
112,T1642,27
...,...,...
373,T0854,1
374,T1171,1
375,T1387,1
376,T1650,1


In [38]:
def keep_frequent_labels(label_str):
    labels = [
        label.strip()
        for label in str(label_str).split(",")
        if label.strip()
    ]

    kept = [
        label
        for label in labels
        if label in frequent_labels
    ]

    return ",".join(kept)


df_stage1 = df_work.copy()
df_stage1["Labels"] = df_stage1["Labels"].apply(keep_frequent_labels)

# Bỏ những dòng không còn frequent label nào
df_stage1 = df_stage1[df_stage1["Labels"].str.len() > 0].reset_index(drop=True)

print("Original dataset:", df_work.shape)
print("Stage 1 dataset:", df_stage1.shape)

df_stage1["label_count"] = df_stage1["Labels"].apply(
    lambda x: len([label for label in str(x).split(",") if label.strip()])
)

print(df_stage1["label_count"].value_counts().sort_index())
display(df_stage1.head())


Original dataset: (22273, 2)
Stage 1 dataset: (20840, 2)
label_count
1    19106
2     1483
3      251
Name: count, dtype: int64


,Cleaned_Text,Labels,label_count
0,$BASE64 encoded payload,T1027,1
1,$INSTDIR File $INSTDIR\o15bmldpqdxcin.dll File...,T1027,1
2,$LogFile Timeline for File Activity DFIR File ...,T1070,1
3,$url = URL_TOKEN This line sets the value of t...,T1105,1
4,%%i in ( WINDOWS_PATH programdata list.txt) do...,T1070,1


In [39]:
def has_rare_label(label_str):
    labels = [
        label.strip()
        for label in str(label_str).split(",")
        if label.strip()
    ]

    return any(label in rare_labels for label in labels)


df_stage2_rare = df_work[df_work["Labels"].apply(has_rare_label)].copy()
df_stage2_rare = df_stage2_rare.reset_index(drop=True)

print("Stage 2 rare dataset:", df_stage2_rare.shape)
display(df_stage2_rare.head())

Stage 2 rare dataset: (1671, 2)


,Cleaned_Text,Labels
0,2FA OTP Timing Skew Cryptography Attacks OTP/2...,T1111
1,51% Attack Blockchain / Web3 Majority Hash Pow...,T1588
2,5G Downgrade via Fake SIB Broadcasting Wireles...,T1431
3,5G SA to NSA Downgrade via PLMN Priority Tampe...,T1431
4,5G to LTE Downgrade and IMSI Catching Wireless...,T1431


In [40]:
stage1_sample_coverage = len(df_stage1) / len(df_work) * 100

print(f"Stage 1 sample coverage: {stage1_sample_coverage:.2f}%")
print("Original samples:", len(df_work))
print("Stage 1 samples:", len(df_stage1))
print("Stage 2 rare-related samples:", len(df_stage2_rare))

Stage 1 sample coverage: 93.57%
Original samples: 22273
Stage 1 samples: 20840
Stage 2 rare-related samples: 1671


In [41]:
total_label_occurrences = label_freq["Sample_Count"].sum()

frequent_label_occurrences = label_freq[
    label_freq["Label"].isin(frequent_labels)
]["Sample_Count"].sum()

rare_label_occurrences = label_freq[
    label_freq["Label"].isin(rare_labels)
]["Sample_Count"].sum()

print("Total label occurrences:", total_label_occurrences)
print("Frequent label occurrences:", frequent_label_occurrences)
print("Rare label occurrences:", rare_label_occurrences)

print(
    f"Frequent label coverage: {frequent_label_occurrences / total_label_occurrences * 100:.2f}%"
)

print(
    f"Rare label coverage: {rare_label_occurrences / total_label_occurrences * 100:.2f}%"
)

Total label occurrences: 24537
Frequent label occurrences: 22825
Rare label occurrences: 1712
Frequent label coverage: 93.02%
Rare label coverage: 6.98%


In [42]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if (Path.cwd().parent / "dataset").exists() else Path.cwd()

PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


stage1_path = PROCESSED_DIR / "attack_dataset_stage1_frequent.csv"
stage2_path = PROCESSED_DIR / "attack_dataset_stage2_rare.csv"


df_stage1[["Cleaned_Text", "Labels"]].to_csv(stage1_path, index=False, encoding="utf-8")
df_stage2_rare[["Cleaned_Text", "Labels"]].to_csv(stage2_path, index=False, encoding="utf-8")



print("Saved Stage 1 dataset:", stage1_path)
print("Saved Stage 2 rare dataset:", stage2_path)


Saved Stage 1 dataset: d:\Truong\FPT\SUMMER2026\AIC211\CTI_ATT&CK\cti_attck\dataset\processed\attack_dataset_stage1_frequent.csv
Saved Stage 2 rare dataset: d:\Truong\FPT\SUMMER2026\AIC211\CTI_ATT&CK\cti_attck\dataset\processed\attack_dataset_stage2_rare.csv


In [43]:
import re
import pandas as pd

df_check = df_final.copy()  # hoặc df_stage1 / df_processed

mitre_id_pattern = r"\bT\d{4}(?:\.\d{3})?\b"

df_check["has_mitre_id_in_text"] = df_check["Cleaned_Text"].str.contains(
    mitre_id_pattern,
    case=False,
    regex=True,
    na=False
)

leak_rows = df_check[df_check["has_mitre_id_in_text"] == True]

print("Rows containing MITRE IDs in text:", len(leak_rows))
print("Percentage:", len(leak_rows) / len(df_check) * 100)

display(leak_rows[["Cleaned_Text", "Labels"]].head(50))

Rows containing MITRE IDs in text: 0
Percentage: 0.0


,Cleaned_Text,Labels


In [44]:
def extract_mitre_ids_from_text(text):
    ids = re.findall(r"\bT\d{4}(?:\.\d{3})?\b", str(text), flags=re.IGNORECASE)
    
    # Map sub-technique về parent
    parent_ids = []
    for x in ids:
        x = x.upper()
        parent = re.match(r"^(T\d{4})(?:\.\d{3})?$", x)
        if parent:
            parent_ids.append(parent.group(1))
    
    return set(parent_ids)


def extract_labels(label_str):
    return set(
        label.strip().upper()
        for label in str(label_str).split(",")
        if label.strip()
    )


df_check["text_mitre_ids"] = df_check["Cleaned_Text"].apply(extract_mitre_ids_from_text)
df_check["label_set"] = df_check["Labels"].apply(extract_labels)

df_check["leaked_label_overlap"] = df_check.apply(
    lambda row: len(row["text_mitre_ids"].intersection(row["label_set"])) > 0,
    axis=1
)

leaked_overlap_rows = df_check[df_check["leaked_label_overlap"] == True]

print("Rows where MITRE ID in text overlaps with Labels:", len(leaked_overlap_rows))
print("Percentage:", len(leaked_overlap_rows) / len(df_check) * 100)

display(
    leaked_overlap_rows[
        ["Cleaned_Text", "Labels", "text_mitre_ids", "label_set"]
    ].head(100)
)

Rows where MITRE ID in text overlaps with Labels: 0
Percentage: 0.0


,Cleaned_Text,Labels,text_mitre_ids,label_set


In [2]:
import pandas as pd

# 1. Đọc file dataset
file_path = "../dataset/processed/attack_dataset_stage1_frequent.csv"
df = pd.read_csv(file_path)

print(f"--- TỔNG QUAN DATASET ---")
print(f"Tổng số dòng: {len(df)}")
print("-" * 50)

# ==============================================================================
# CÁCH 1: Kiểm tra theo TỔ HỢP NHÃN (Exact Label Combinations)
# (Coi mỗi tổ hợp như 'T1059,T1105' là 1 class độc lập - Multiclass)
# ==============================================================================
label_counts = df["Labels"].value_counts()
under_30_exact = label_counts[label_counts < 30]

print("1. KIỂM TRA THEO TỔ HỢP NHÃN (Exact Combinations):")
print(f" - Tổng số tổ hợp nhãn độc lập: {len(label_counts)}")
print(f" - Số tổ hợp nhãn có dưới 30 mẫu: {len(under_30_exact)}")
if len(under_30_exact) > 0:
    print("\nDanh sách các tổ hợp nhãn dưới 30 mẫu (Top 10 ít nhất):")
    print(under_30_exact.tail(10))
print("-" * 50)

# ==============================================================================
# CÁCH 2: Kiểm tra theo TỪNG KỸ THUẬT ĐƠN LẺ (Individual Techniques)
# (Tách các nhãn bị gộp bởi dấu phẩy để đếm tần suất thực tế - Multi-label)
# ==============================================================================
# Tách chuỗi theo dấu phẩy, loại bỏ khoảng trắng dư thừa và làm phẳng (explode)
individual_labels = (
    df["Labels"].astype(str).str.split(",").explode().str.strip()
)
ind_counts = individual_labels.value_counts()
under_30_ind = ind_counts[ind_counts < 30]

print("2. KIỂM TRA THEO TỪNG KỸ THUẬT ĐƠN LẺ (Exploded Techniques):")
print(f" - Tổng số kỹ thuật (Technique ID) độc lập: {len(ind_counts)}")
print(f" - Số kỹ thuật có dưới 30 mẫu: {len(under_30_ind)}")
if len(under_30_ind) > 0:
    print("\nDanh sách các kỹ thuật dưới 30 mẫu:")
    print(under_30_ind)
else:
    print(
        " => Tất cả các kỹ thuật đơn lẻ đều có từ 30 mẫu trở lên trong dataset!"
    )


--- TỔNG QUAN DATASET ---
Tổng số dòng: 20840
--------------------------------------------------
1. KIỂM TRA THEO TỔ HỢP NHÃN (Exact Combinations):
 - Tổng số tổ hợp nhãn độc lập: 884
 - Số tổ hợp nhãn có dưới 30 mẫu: 775

Danh sách các tổ hợp nhãn dưới 30 mẫu (Top 10 ít nhất):
Labels
T1012,T1071          1
T1041,T1071,T1105    1
T1190,T1595          1
T1208,T1602          1
T1068,T1078,T1530    1
T1059,T1601          1
T1055,T1057,T1059    1
T1003,T1053,T1078    1
T1200,T1592          1
T1082,T1106,T1573    1
Name: count, dtype: int64
--------------------------------------------------
2. KIỂM TRA THEO TỪNG KỸ THUẬT ĐƠN LẺ (Exploded Techniques):
 - Tổng số kỹ thuật (Technique ID) độc lập: 108
 - Số kỹ thuật có dưới 30 mẫu: 0
 => Tất cả các kỹ thuật đơn lẻ đều có từ 30 mẫu trở lên trong dataset!
